# 05. Union-Find (Disjoint Set Union)

A structure that tracks **groups of connected items** and answers two questions almost in O(1):
- `find(x)`: which group is `x` in?
- `union(a, b)`: merge the groups of `a` and `b`.

**Learning order:** REAL LIFE → IDEA → OPTIMIZATIONS → CODE → PROVINCES → REDUNDANT CONNECTION → VALID TREE → EQUATIONS → INTERVIEW THINKING

## 1. Real life

- **Friend circles:** when two people become friends, their circles merge. "Are A and B in the same circle?"
- **Network connectivity:** are two computers connected, directly or indirectly?
- **Kruskal's MST**, **image segmentation**, **cycle detection** in undirected graphs.

DSU only supports **merging**, not splitting. That is why it is so fast.

## 2. Idea

Every group is a **tree**, represented by its **root** (the representative). `parent[x]` points toward the root; a root points to itself.

- `find(x)`: follow parents to the root.
- `union(a, b)`: make one root the parent of the other.

**Two optimizations** make it nearly O(1) (amortized `O(α(n))`, the inverse Ackermann function, at most ~4 in practice):
1. **Path compression:** while finding, make nodes point closer to the root.
2. **Union by size/rank:** attach the **smaller** tree under the bigger one, so trees stay shallow.

In [1]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.count = n                          # number of components

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]     # path compression (halving)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                        # already connected
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra                    # attach the smaller tree under the bigger one
        self.size[ra] += self.size[rb]
        self.count -= 1
        return True

d = DSU(6)
d.union(0, 1); d.union(1, 2); d.union(3, 4)
print(d.find(0) == d.find(2), d.find(0) == d.find(3), d.count)   # True False 3

True False 3


## 3. Number of Provinces

`isConnected[i][j] = 1` means cities `i` and `j` are directly connected. Provinces = connected components. Union every connected pair; the answer is `dsu.count`.

In [2]:
def find_circle_num(is_connected):
    n = len(is_connected)
    dsu = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if is_connected[i][j]:
                dsu.union(i, j)
    return dsu.count

print(find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]))   # 2

2


## 4. Redundant Connection

A tree plus **one extra edge**. Find the edge that creates the cycle. Add edges one by one. The first edge whose endpoints are **already connected** (`union` returns `False`) is the answer.

In [3]:
def find_redundant_connection(edges):
    dsu = DSU(len(edges) + 1)                       # nodes are 1..n
    for u, v in edges:
        if not dsu.union(u, v):
            return [u, v]

print(find_redundant_connection([[1, 2], [1, 3], [2, 3]]))                          # [2, 3]
print(find_redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))          # [1, 4]

[2, 3]
[1, 4]


## 5. Graph Valid Tree

`n` nodes form a tree iff: exactly `n - 1` edges **and** no cycle (or connected). Using DSU: if any `union` fails, there is a cycle; at the end `count` must be `1`.

In [4]:
def valid_tree(n, edges):
    if len(edges) != n - 1:
        return False
    dsu = DSU(n)
    for u, v in edges:
        if not dsu.union(u, v):
            return False
    return dsu.count == 1

print(valid_tree(5, [[0, 1], [0, 2], [0, 3], [1, 4]]), valid_tree(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]]))   # True False

True False


## 6. Satisfiability of Equality Equations

Equations like `a==b`, `b!=c`. First **union** all `==` pairs. Then every `!=` pair must be in **different** groups.

In [5]:
def equations_possible(equations):
    dsu = DSU(26)
    for eq in equations:
        if eq[1] == "=":
            dsu.union(ord(eq[0]) - 97, ord(eq[3]) - 97)
    for eq in equations:
        if eq[1] == "!" and dsu.find(ord(eq[0]) - 97) == dsu.find(ord(eq[3]) - 97):
            return False
    return True

print(equations_possible(["a==b", "b!=a"]), equations_possible(["a==b", "b==c", "a==c"]), equations_possible(["a==b", "b!=c", "c==a"]))   # False True False

False True False


## 7. DSU vs BFS/DFS for connectivity

| | DSU | BFS / DFS |
|---|---|---|
| edges arrive **one by one** (dynamic) | **great** | recompute each time |
| static graph, one query | fine | fine |
| need paths or distances | no | yes |
| cycle detection (undirected) | `union` returns `False` | parent tracking |

**Variants:** `parent` as dict for arbitrary labels (strings), size tracking for "largest group", weighted DSU for ratios (Evaluate Division).

## 8. Interview thinking

**Signals:** "are X and Y connected", "number of groups", "merge accounts/sets", "which edge forms a cycle", "edges added over time".

**Say:** "I'll use Union-Find with path compression and union by size: near O(1) per operation, O(n) space."

**Common mistakes:** forgetting to call `find` on both nodes in `union` (comparing raw parents), missing path compression (O(n) chains), off-by-one with 1-indexed nodes (allocate `n + 1`), not decrementing the component count only on a successful merge.

## Quick Revision

- `find` with path compression, `union` by size. `union` returns `False` if already connected.
- Components = `count`. Cycle = failed union.
- Tree check: `n - 1` edges and no cycle.
- Equations: union `==` first, check `!=` after.
- Best for incremental connectivity.